# Image Captioning on the Flicker8k Dataset

This notebook provides you with a rough structure for the Mini-Challenge on the topic of Image Captioning from the Deep Learning MSE course. Please make sure to follow the Project Setup steps to ensure reproducable results. Additionally familiarize yourself with the grading structure and project submission guidelines in the _DL-MPW-ImgCaptioning.pdf_ document. There you'll find all the information you should need. 

If you have any questions please contact roman.studer@fhnw.ch

###  Team Name and Members
Mats Ricke
Tiziano Schacht
Christoph Walser


### Some Guidelines

- You are allowed to use generative models (e.g. ChatGPT, Claude etc.) for code completion or suggestions. However, we require you to perfectly understand and control the code. 
- You are **not** allowed to use the same or similar generative models for providing reflection on the achieved results. __You__ must reflect about what you see and build your own opinion about it.
- You are allowed to create additional python scripts to outsource code snippets to improve readability.

## Project Setup


### 1. Create the virtual environment

Create a new environment and install the necessary dependencies listed in `requirements.txt`. Make sure you can import the libraries in the code cell below and that yout pytorch installation can run on your GPU.

In [ ]:
# imports
import textwrap

import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision

from dataloader import (
    build_tokenizer_from_split,
    create_caption_dataloader,
    load_split_dataframe,
    build_captions_map
)
from tokenizer import tokenize

print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)

import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

import wandb

### 2. Download the dataset
The Flickr8k dataset used in this challenge is available under [Flickr 8k Dataset - Kaggle](https://www.kaggle.com/datasets/adityajn105/flickr8k). In the zip folder on Moodle you find the basic structure with `captions.txt` and the `data/splits`-folder, but without images in the `data/ìmages`-folder. Make sure to download all the images and placing them in the folder `data/images/`. 
The train test split is already provided for you and handled by the provided `dataloader.py` script. Test and train split image lists and corresponding captions are located in the `data/splits/` folder. Don't modify this split.

<img src="data_structure.png"/>

## Dataloader, Tokenizer

The repository includes a `Dataset`-class (`Flickr8kCaptionsDatasetBase`) and `DataLoader`'s that are based on this specific structure of the data directory. You can use `create_caption_dataloader` for obtaining instances for the train and test data loaders. You can configure it with the `tokenizer` provided in the repo. The `tokenizer` performs some preprocessing steps of the captions. It also creates a vocabulary from the training data and then provides a mapping from the words to tokens. Note that there are also some special tokens introduced that are important when processing the captions during training or generating the captions during inference:
| Word    | ID | Role |
| --------| ---|----- |
| `<pad>`  | 0 | padding for filling up sequence to a given length in a mini-batch. | 
| `<bos>`  | 1 | beginning of sentence - each tokenized sentence should start with `<bos>`, i.e. also the generated ones. | 
| `<eos>`  | 2 | end of sentence - for fixed length captions will signal that the rest will consist of `<pad>`-tokens only. | 
| `<unk>`  | 3 | unknown - not available in vocabulary | 

Furthermore, specify your transforms used for preprocessing and augmenting the images and pass them to the `create_caption_dataloader`-method.
The mini-batches created by the dataloader have the following structure: `(images, captions, lengths, image_ids, raw_captions)`. Captions are built from the training vocabulary only, and `lengths` may let you mask padding tokens in the loss.


In [14]:
# Transforms
from torchvision import transforms

IMAGENET_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def _convert_to_rgb(image: Image.Image) -> Image.Image:
    return image.convert("RGB")

def build_train_transforms() -> transforms.Compose:
    return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            # START - MODIFY / EXTEND THIS PART
            transforms.Resize((IMAGENET_SIZE,IMAGENET_SIZE)),
            # END - MODIFY / EXTEND THIS PART
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

def build_test_transforms() -> transforms.Compose:
    return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.Resize((IMAGENET_SIZE,IMAGENET_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

def denormalize_image(image: torch.Tensor) -> torch.Tensor:
    """Undo ImageNet normalization for visualization."""
    mean_tensor = image.new_tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std_tensor = image.new_tensor(IMAGENET_STD).view(-1, 1, 1)
    return (image * std_tensor + mean_tensor).clamp(0, 1)

In [15]:
# TODO: ADJUST PATH TO data_dir IF NEEDED
data_dir = "./data/"
# TODO: SET REASONABLE VALUES FOR THE FOLLOWING PARAMETERS
MIN_WORD_FREQ = 5
MAX_LEN = 40
BATCH_SIZE = 32

In [ ]:
# TODO: POSSIBLY USE YOUR OWN TOKENIZER AND SET IT TO tokenizer=YourTokenizer(...)
# Make sure to build the vocabulary only from training captions to avoid test leakage.

tokenizer = build_tokenizer_from_split(split="train", data_dir = data_dir, min_freq=MIN_WORD_FREQ)

train_loader = create_caption_dataloader(
    split="train", 
    data_dir = data_dir, 
    tokenizer=tokenizer, 
    transform = build_train_transforms(),
    batch_size = BATCH_SIZE, 
    max_len=MAX_LEN, 
    caption_sampling="all")
test_loader = create_caption_dataloader(
    split="test", 
    data_dir = data_dir, 
    tokenizer=tokenizer, 
    transform = build_test_transforms(),
    batch_size = BATCH_SIZE, 
    max_len=MAX_LEN, 
    caption_sampling="random")

images, captions, lengths, image_ids, raw_captions = next(iter(train_loader))
# images:    (B, 3, 224, 224)  — batch of normalized image tensors
# captions:  (B, 40)            — token IDs, padded to MAX_LEN
# lengths:   (B,)               — real length of each caption (incl. <bos>/<eos>)
# image_ids: list of B filenames
# raw_captions: list of B original strings
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Number of training examples: {len(train_loader.dataset):,}")
print(f"Batch image tensor shape: {tuple(images.shape)}")
print(f"Batch caption tensor shape: {tuple(captions.shape)}")
print(f"Caption lengths in first batch: {lengths.tolist()}")

images, captions, lengths, image_ids, raw_captions = next(iter(test_loader))

print(f"Number of test examples: {len(test_loader.dataset):,}")
print(f"Batch image tensor shape: {tuple(images.shape)}")
print(f"Batch caption tensor shape: {tuple(captions.shape)}")
print(f"Caption lengths in first batch: {lengths.tolist()}")

In [ ]:
num_examples = 12
n = min(num_examples, len(images))

fig, axes = plt.subplots(
    nrows=n,
    ncols=2,
    figsize=(16, 4 * n),
    gridspec_kw={"width_ratios": [1.1, 2.2]},
)

if n == 1:
    axes = [axes]

for idx in range(n):
    image = denormalize_image(images[idx].detach().cpu()).permute(1, 2, 0)

    encoded_caption = captions[idx, : int(lengths[idx])].tolist()
    tokenized_caption = tokenize(raw_captions[idx])
    decoded_caption = tokenizer.decode(encoded_caption, skip_special_tokens=False)

    ax_image, ax_text = axes[idx]
    ax_image.imshow(image)
    ax_image.set_title(image_ids[idx])
    ax_image.axis("off")

    caption_details = "\n\n".join(
        [
            f"Raw caption:\n{textwrap.fill(raw_captions[idx], width=70)}",
            f"Tokenized caption:\n{textwrap.fill(str(tokenized_caption), width=70)}",
            f"Encoded caption:\n{textwrap.fill(str(encoded_caption), width=70)}",
            f"Decoded caption:\n{textwrap.fill(decoded_caption, width=70)}",
        ]
    )

    ax_text.axis("off")
    ax_text.text(
        0,
        1,
        caption_details,
        va="top",
        ha="left",
        fontsize=11,
        family="monospace",
    )

plt.tight_layout()
plt.show()

## Shared Utilities

Use this section for shared config and helper functions reused across all experiments. For example define a common training loop, BLEU computation or visualization helpers here. This way your model models stay directly comparable.


In [ ]:
# Define shared config, helper functions, a fixed evaluation subset, and BLEU computation.
DEVICE = None
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else: 
    DEVICE = torch.device("cpu")
DEVICE = torch.device("cpu")

print("device:", DEVICE)
# further helper functionality:

def init_wandb_run(run_name, config, entity= "chris-wals-zhaw", project="DL_MPW_ImageCaptioning", tags=None):
    """
    Start a W&B run. Call once per training session.
    
    Args:
        run_name: e.g. "show-and-tell-v1"
        config: dict of hyperparameters (will appear in the run's "Config" tab). example:
            {
                "learning_rate": 0.02,
                "architecture": "CNN",
                "dataset": "CIFAR-100",
                "epochs": 10,
            }
        project: groups related runs together
        tags: list of strings for filtering, e.g. ["baseline"], ["attention"]
    """
    return wandb.init(
        entity=entity,
        project=project,
        name=run_name,
        config=config,
        tags=tags or [],
        reinit=True,  # lets you start multiple runs in one notebook session
    )
def build_test_references(data_dir):
    """
    For BLEU we need all 5 reference captions per test image.
    Returns: dict image_id -> list of 5 tokenized captions.
    """
    test_df = load_split_dataframe("test", data_dir)
    cap_map = build_captions_map(test_df)
    return {img_id: [tokenize(c) for c in caps] for img_id, caps in cap_map.items()}


def train_one_epoch(model, loader, optimizer, criterion, device):
    """One pass over the training set with teacher forcing."""
    model.train()
    total_loss = 0.0
    for images, captions, lengths, _ids, _raw in loader:
        images = images.to(device)
        captions = captions.to(device)
 
        # Teacher forcing: model sees true previous tokens, predicts next ones.
        inputs = captions[:, :-1]      # everything except last token
        targets = captions[:, 1:]      # everything except first token (shifted)
 
        logits = model(images, inputs)  # (B, T, vocab_size)
 
        # CrossEntropyLoss wants (N, C) and (N,)
        # ignore_index=pad_idx skips padding positions automatically.
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def evaluate_loss(model, loader, criterion, device):
    """Compute average loss on the test set, no gradient updates."""
    model.eval()
    total_loss = 0.0
    for images, captions, lengths, _ids, _raw in loader:
        images = images.to(device)
        captions = captions.to(device)
        inputs = captions[:, :-1]
        targets = captions[:, 1:]
        logits = model(images, inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        total_loss += loss.item()
    return total_loss / len(loader)

def train_model(model, train_loader, test_loader, tokenizer, device, num_epochs=15, lr=4e-4):
    """Train for num_epochs and return loss history."""
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr
    )
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
 
    train_losses, val_losses = [], []
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = evaluate_loss(model, test_loader, criterion, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f}")
 
    return {"train_loss": train_losses, "val_loss": val_losses}
 
 
def plot_loss_curves(history, title=""):
    plt.figure(figsize=(7, 4))
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="val")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
 
 
@torch.no_grad()
def generate_captions(model, loader, tokenizer, device, max_len=35):
    """Generate one caption per unique test image."""
    model.eval()
    predictions = {}
    for images, _captions, _lengths, image_ids, _raw in loader:
        images = images.to(device)
        generated = model.generate(images, max_len=max_len)  # (B, max_len)
 
        for i, img_id in enumerate(image_ids):
            if img_id in predictions:
                continue
            text = tokenizer.decode(generated[i].tolist(), skip_special_tokens=True)
            predictions[img_id] = text.split()
    return predictions
 
 
def compute_bleu(predictions, references_map):
    """Compute BLEU-1 through BLEU-4."""
    hypotheses, references = [], []
    for img_id, hyp_tokens in predictions.items():
        if img_id in references_map:
            hypotheses.append(hyp_tokens)
            references.append(references_map[img_id])
 
    smooth = SmoothingFunction().method1
    return {
        "BLEU-1": corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smooth),
        "BLEU-2": corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smooth),
        "BLEU-3": corpus_bleu(references, hypotheses, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smooth),
        "BLEU-4": corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth),
    }
 
 
def evaluate_model(model, test_loader, tokenizer, device, references_map):
    """Compute perplexity and BLEU on the test set."""
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
    val_loss = evaluate_loss(model, test_loader, criterion, device)
    perplexity = math.exp(val_loss)
 
    predictions = generate_captions(model, test_loader, tokenizer, device)
    bleu = compute_bleu(predictions, references_map)
 
    print(f"Perplexity: {perplexity:.2f}")
    for k, v in bleu.items():
        print(f"  {k}: {v:.4f}")
    return perplexity, bleu
 
 
def show_predictions(model, loader, tokenizer, device, references_map, num=6, denormalize_fn=None):
    """Display a few images alongside their generated and reference captions."""
    model.eval()
    images, _, _, image_ids, _ = next(iter(loader))
    images = images.to(device)
    with torch.no_grad():
        generated = model.generate(images, max_len=35)
 
    n = min(num, images.size(0))
    fig, axes = plt.subplots(n, 2, figsize=(14, 3.5 * n), gridspec_kw={"width_ratios": [1, 2]})
    if n == 1:
        axes = [axes]
 
    for i in range(n):
        img = images[i].cpu()
        if denormalize_fn is not None:
            img = denormalize_fn(img)
        img = img.permute(1, 2, 0).numpy().clip(0, 1)
 
        gen_text = tokenizer.decode(generated[i].tolist(), skip_special_tokens=True)
        refs = references_map.get(image_ids[i], [])
        ref_text = "\n".join(f"  - {' '.join(r)}" for r in refs[:3])
 
        axes[i][0].imshow(img); axes[i][0].axis("off"); axes[i][0].set_title(image_ids[i], fontsize=9)
        axes[i][1].axis("off")
        axes[i][1].text(0, 1, f"GENERATED:\n  {gen_text}\n\nREFERENCES:\n{ref_text}",
                        va="top", fontsize=10, family="monospace")
    plt.tight_layout()
    plt.show()






## Model 1 - Show and Tell

Implement the model described in the [_Show and Tell_](https://arxiv.org/abs/1411.4555) paper . Record the training behavior, inspect a few generated captions on held-out images, and report BLEU on your fixed evaluation subset.

In [25]:
# TODO Parameters for the Show and Tell architecture  
ENCODER_DIM = 512
EMBED_DIM = 256
DECODER_DIM = 512
DROPOUT = 0.5
FREEZE_ENCODER = True

In [26]:
# TODO Implement the Show and Tell model classes and any model-specific setup here.


class ShowAndTellEncoder(nn.Module):
    """Pretrained ResNet18 -> one 256-dim vector per image."""
 
    def __init__(self, embed_dim, freeze=True):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Drop the final classification layer; keep everything else.
        # Output of this backbone: (B, 512, 1, 1) — one 512-dim vector per image.
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.project = nn.Linear(512, embed_dim)
 
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
 
    def forward(self, images: torch.Tensor):
        # images: (B, 3, 224, 224)
        features = self.backbone(images)          # (B, 512, 1, 1)
        features = features.flatten(1)             # (B, 512)
        return self.project(features)              # (B, embed_dim)
 
 
class ShowAndTellCaptioner(nn.Module):
    """
    The image vector is fed as the FIRST input to the LSTM.
    Then word embeddings follow. The LSTM has to remember the image
    using only its hidden state.
    """
 
    def __init__(self, vocab_size, embed_dim, decoder_dim, encoder,
                 pad_idx, bos_idx, dropout=0.5):
        super().__init__()
        self.encoder = encoder
        self.bos_idx = bos_idx
 
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, decoder_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(decoder_dim, vocab_size)
 
    def forward(self, images, captions: torch.Tensor):
        # images: (B, 3, 224, 224); captions: (B, T) input tokens
        img_feat = self.encoder(images).unsqueeze(1)   # (B, 1, embed_dim)
        word_emb = self.embedding(captions)             # (B, T, embed_dim)
 
        # Stick the image in front of the words: total length T+1
        inputs = torch.cat([img_feat, word_emb], dim=1)  # (B, T+1, embed_dim)
 
        lstm_out, _ = self.lstm(inputs)                  # (B, T+1, decoder_dim)
        # Drop the output at the image-step (position 0). We only want
        # predictions that follow real word inputs. Result: (B, T, vocab_size)
        logits = self.fc(self.dropout(lstm_out[:, 1:]))
        return logits
 
    @torch.no_grad()
    def generate(self, images: torch.Tensor, max_len=35):
        """Greedy decoding: at each step pick the most likely next word."""
        self.eval()
        B = images.size(0)
        device = images.device
 
        img_feat = self.encoder(images).unsqueeze(1)    # (B, 1, embed_dim)
        # Feed the image first to set up the LSTM's hidden state
        _, state = self.lstm(img_feat)
 
        # Start generation from <bos>
        current = torch.full((B, 1), self.bos_idx, dtype=torch.long, device=device)
        outputs = []
        for _ in range(max_len):
            emb = self.embedding(current)               # (B, 1, embed_dim)
            out, state = self.lstm(emb, state)          # (B, 1, decoder_dim)
            logits = self.fc(out.squeeze(1))            # (B, vocab_size)
            current = logits.argmax(dim=-1, keepdim=True)  # (B, 1)
            outputs.append(current)
        return torch.cat(outputs, dim=1)                # (B, max_len)


In [ ]:
# TODO: Add the training loop, caption generation, evaluation and metric calculation logic for Model 1 here and in the following cells.
references_map = build_test_references(data_dir)
encoder1 = ShowAndTellEncoder(EMBED_DIM, freeze=FREEZE_ENCODER).to(DEVICE)
model1 = ShowAndTellCaptioner(
       vocab_size=len(tokenizer), embed_dim=EMBED_DIM, decoder_dim=DECODER_DIM,
       encoder=encoder1, pad_idx=tokenizer.pad_idx, bos_idx=tokenizer.bos_idx,
       dropout=DROPOUT,
).to(DEVICE)

history1 = train_model(model1, train_loader, test_loader, tokenizer, DEVICE, num_epochs=15)
plot_loss_curves(history1, title="Show and Tell")
evaluate_model(model1, test_loader, tokenizer, DEVICE, references_map)
show_predictions(model1, test_loader, tokenizer, DEVICE, references_map,
                denormalize_fn=denormalize_image)

Briefly summarize what happened during training, which captions looked convincing or weak, and what the BLEU score does or does not capture for this model.


<span style="color: red;"><strong>TODO:</strong> Write your reflection on Model 1 here.</span>


## Model 2 - Show, Attend and Tell

Now implement the continuation of the Show and Tell paper. An attention-based captioning model described in the paper [_Show, Attend and Tell_](https://arxiv.org/abs/1502.03044). Use the same test data subset and BLEU setup as before, then compare how attention changes the training behavior and generated captions.


In [ ]:
# TODO Parameters for the Show, Attend and Tell Architecture
ENCODER_DIM = 512
EMBED_DIM = 256
DECODER_DIM = 512
DROPOUT = 0.5
FREEZE_ENCODER = True
ATTENTION_DIM = 256

In [ ]:
class ShowAttendTellEncoder(nn.Module):
    """
    Same ResNet18, but we KEEP the spatial structure.
    Output: (B, 49, 512) — 49 regions, each described by 512 numbers.
    """
 
    def __init__(self, freeze=True):
        super().__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Drop avgpool AND fc — we want the 7x7 spatial map.
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
 
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
 
    def forward(self, images):
        features = self.backbone(images)               # (B, 512, 7, 7)
        B, C, H, W = features.shape
        # Rearrange to (B, 49, 512): 49 spatial positions, 512-dim each
        return features.permute(0, 2, 3, 1).reshape(B, H * W, C)
 
 
class AdditiveAttention(nn.Module):
    """
    Bahdanau attention.
 
    Given the 49 region vectors and the decoder's current hidden state,
    decide which regions to look at right now.
    """
 
    def __init__(self, encoder_dim, decoder_dim, attention_dim):
        super().__init__()
        self.encoder_proj = nn.Linear(encoder_dim, attention_dim)
        self.decoder_proj = nn.Linear(decoder_dim, attention_dim)
        self.score = nn.Linear(attention_dim, 1)
 
    def forward(self, encoder_out, decoder_hidden):
        # encoder_out:    (B, 49, encoder_dim)
        # decoder_hidden: (B, decoder_dim)
        enc = self.encoder_proj(encoder_out)                   # (B, 49, A)
        dec = self.decoder_proj(decoder_hidden).unsqueeze(1)   # (B, 1, A)
        scores = self.score(torch.tanh(enc + dec)).squeeze(-1) # (B, 49)
        alpha = F.softmax(scores, dim=1)                       # (B, 49) sums to 1
        context = (alpha.unsqueeze(-1) * encoder_out).sum(dim=1)  # (B, encoder_dim)
        return context, alpha
 
 
class ShowAttendTellCaptioner(nn.Module):
    """
    At each step:
      1. Use the current LSTM hidden state to compute attention weights.
      2. Take a weighted average of the 49 regions -> context vector.
      3. Feed [word_embedding, context_vector] to one LSTM step.
      4. Predict the next word.
    """
 
    def __init__(self, vocab_size, embed_dim, encoder_dim, decoder_dim,
                 attention_dim, encoder, pad_idx, bos_idx, dropout=0.5):
        super().__init__()
        self.encoder = encoder
        self.bos_idx = bos_idx
        self.vocab_size = vocab_size
 
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.attention = AdditiveAttention(encoder_dim, decoder_dim, attention_dim)
        # LSTM input dim = embed_dim + encoder_dim (word + context concatenated)
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        # Initial hidden/cell state derived from mean of encoder features
        self.init_h = nn.Linear(encoder_dim, decoder_dim)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(decoder_dim, vocab_size)
 
    def _init_state(self, encoder_out):
        mean_feat = encoder_out.mean(dim=1)            # (B, encoder_dim)
        h = torch.tanh(self.init_h(mean_feat))
        c = torch.tanh(self.init_c(mean_feat))
        return h, c
 
    def forward(self, images, captions):
        encoder_out = self.encoder(images)             # (B, 49, encoder_dim)
        B, L, _ = encoder_out.shape
        T = captions.size(1)
 
        h, c = self._init_state(encoder_out)
        embeds = self.embedding(captions)              # (B, T, embed_dim)
 
        logits_all = images.new_zeros(B, T, self.vocab_size)
        for t in range(T):
            context, _alpha = self.attention(encoder_out, h)
            lstm_input = torch.cat([embeds[:, t, :], context], dim=1)  # (B, embed+enc)
            h, c = self.lstm_cell(lstm_input, (h, c))
            logits_all[:, t, :] = self.fc(self.dropout(h))
        return logits_all
 
    @torch.no_grad()
    def generate(self, images, max_len=35):
        self.eval()
        encoder_out = self.encoder(images)
        B = encoder_out.size(0)
        device = images.device
 
        h, c = self._init_state(encoder_out)
        current = torch.full((B,), self.bos_idx, dtype=torch.long, device=device)
 
        outputs = []
        for _ in range(max_len):
            emb = self.embedding(current)              # (B, embed_dim)
            context, _alpha = self.attention(encoder_out, h)
            lstm_input = torch.cat([emb, context], dim=1)
            h, c = self.lstm_cell(lstm_input, (h, c))
            logits = self.fc(h)
            current = logits.argmax(dim=-1)            # (B,)
            outputs.append(current)
        return torch.stack(outputs, dim=1)             # (B, max_len)


In [ ]:
# TODO: Add the training loop, caption generation, evaluation and metric calculation logic for Model 2 here and in the following cells.
encoder2 = ShowAttendTellEncoder(freeze=FREEZE_ENCODER).to(DEVICE)
model2 = ShowAttendTellCaptioner(
       vocab_size=len(tokenizer), embed_dim=EMBED_DIM, encoder_dim=ENCODER_DIM,
       decoder_dim=DECODER_DIM, attention_dim=ATTENTION_DIM, encoder=encoder2,
       pad_idx=tokenizer.pad_idx, bos_idx=tokenizer.bos_idx, dropout=DROPOUT,
).to(DEVICE)

history2 = train_model(model2, train_loader, test_loader, tokenizer, DEVICE,
                     num_epochs=15, lr=4e-4)
plot_loss_curves(history2, title="Show, Attend and Tell")
evaluate_model(model2, test_loader, tokenizer, DEVICE, references_map)
show_predictions(model2, test_loader, tokenizer, DEVICE, references_map,
              denormalize_fn=denormalize_image)


Briefly summarize how this model behaved during training, where attention seemed to help or fail, and how the BLEU score compares to Model 2.


<span style="color: red;"><strong>TODO:</strong> Write your reflection on Model 2 here.</span>


## Additional Task 1: Option ...

Formulate and rationalize a hypothesis.

Carry through an experiment for testing this hypothesis.

Describe the outcome and carefully evaluate and reflect about it. 

## Additional Task 2: Option ....

Formulate and rationalize a hypothesis.

Carry through an experiment for testing this hypothesis.

Describe the outcome and carefully evaluate and reflect about it. 